In [1]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats

solver = 'appsi_highs'
 
import pyomo.environ as pyo
SOLVER = pyo.SolverFactory(solver)
assert SOLVER.available(), f"Solver {solver} is not available."
# problem data
c = 10
p = 25
h = 3


def SeafoodStockDeterministic():
    model = pyo.ConcreteModel(
        "Seafood distribution center - Deterministic average demand"
    )

    # key parameter for possible parametric study
    model.mean_demand = pyo.Param(initialize=100, mutable=True)

    # first stage variables and expressions
    model.x = pyo.Var(domain=pyo.NonNegativeReals)

    @model.Expression()
    def first_stage_profit(m):
        return -c * model.x

    # second stage variables, constraints, and expressions
    model.y = pyo.Var(domain=pyo.NonNegativeReals)
    model.z = pyo.Var(domain=pyo.NonNegativeReals)

    @model.Constraint()
    def cant_sell_fish_i_dont_have(m):
        return m.y <= m.mean_demand

    @model.Constraint()
    def fish_do_not_disappear(m):
        return m.y + m.z == m.x

    @model.Expression()
    def second_stage_profit(m):
        return p * m.y - h * m.z

    # objective
    @model.Objective(sense=pyo.maximize)
    def total_profit(m):
        return m.first_stage_profit + m.second_stage_profit

    return model


model = SeafoodStockDeterministic()
result = SOLVER.solve(model)
assert result.solver.status == "ok"
assert result.solver.termination_condition == "optimal"

print(
    f"Optimal solution for determistic demand equal to the average demand = {model.x():.1f} tons"
)
print(f"Optimal deterministic profit = {model.total_profit():.0f}€")

Optimal solution for determistic demand equal to the average demand = 100.0 tons
Optimal deterministic profit = 1500€


In [2]:
model.y()

100.0

In [3]:
model.z()

0.0

In [ ]:
import numpy as np
import pyomo.environ as pyo

np.random.seed(1956)

# ── First Stage Constants ──────────────────────────────────────────────────────
c = np.array([6, 14, 27]) # Installation (allocation) costs
p = np.array([1,  5,  3]) # Space (power) needed
B = 1000 # Budget available
S = 100  # Space available

# ── Second Stage Constants ─────────────────────────────────────────────────────
S = 3 # Number of scenarios for tree
d_xi = np.array([[60, 6, 12],  # low
                 [70, 8, 15],  # medium
                 [75, 10, 18]]) # high
r = np.array([7, 18, 30])    # Costs of each resource
#p = np.array([1.0 / S] * S)  # Uniform scenario probabilities
#p = np.array([0.3, 0.5, 0.2 ]) # gives 0, 2, 10 and cost = 1562€
p = np.array([0.1, 0.8, 0.1 ]) # gives 4, 2, 10 and cost = 829€

# ── Index Sets ─────────────────────────────────────────────────────────────────
RESOURCES  = [0, 1, 2]   # CPU, GPU, TPU
SCENARIOS  = list(range(S))

# ── Model ──────────────────────────────────────────────────────────────────────
m = pyo.ConcreteModel(name="2SP-TREE")

# ── Variables ──────────────────────────────────────────────────────────────────
m.x = pyo.Var(RESOURCES, domain=pyo.NonNegativeIntegers)          # First stage
m.y = pyo.Var(RESOURCES, SCENARIOS, domain=pyo.NonNegativeIntegers)  # Recourse

# ── Objective ──────────────────────────────────────────────────────────────────
m.obj = pyo.Objective(
    expr=(
        sum(c[i] * m.x[i] for i in RESOURCES)
      + sum(p[s] * sum(r[i] * m.y[i, s] for i in RESOURCES)
            for s in SCENARIOS
          )
    ),
    sense=pyo.minimize
)

# ── Constraints ────────────────────────────────────────────────────────────────
## Installation budget
m.budget = pyo.Constraint(
    expr=sum(c[i] * m.x[i] for i in RESOURCES) <= B
)

## Space (power) available
m.space = pyo.Constraint(
    expr=sum(p[i] * m.x[i] for i in RESOURCES) <= S
)

## Meet demand in every scenario  (x_i + y_{i,s} >= d_{i,s})
m.demand = pyo.Constraint(
    RESOURCES, SCENARIOS,
    rule=lambda m, i, s: m.x[i] + m.y[i, s] >= d_xi[i, s]
)

# ── Solve ──────────────────────────────────────────────────────────────────────
solver = pyo.SolverFactory("gurobi")        # swap for "glpk",  "cbc" or "gurobi" freely
results = solver.solve(m, tee=False)        # tee=True for verbose output

# ── Results ────────────────────────────────────────────────────────────────────
if results.solver.termination_condition == pyo.TerminationCondition.optimal:
    x_opt = np.array([pyo.value(m.x[r]) for r in RESOURCES])
    obj_val = pyo.value(m.obj)
    print(
        f"Our server farm will cost €{obj_val:,.2f}. "
        f"\nWe should buy/allocate {int(x_opt[0])} CPUs, "
        f"{int(x_opt[1])} GPUs, and "
        f"{int(x_opt[2])} TPUs."
    )
else:
    print("Solver did not find an optimal solution:",
          results.solver.termination_condition)

Our server farm will cost €829.20. 
We should buy/allocate 4 CPUs, 2 GPUs, and 10 TPUs.
